In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

# Day 2 Lab: Combinational Building Blocks

> **Week 1, Session 2** · Accelerated HDL for Digital System Design · UCF ECE

## Overview

| | |
|---|---|
| **Duration** | ~2 hours |
| **Prerequisites** | Pre-class video (45 min): data types, vectors, operators, continuous assignment |
| **Deliverable** | 7-segment display showing button states as hex digit, programmed on board |
| **Tools** | Yosys, nextpnr-ice40, icepack, iceprog |

## Learning Objectives

| SLO | Description |
|-----|-------------|
| 2.1 | Declare and manipulate vectors using bit selection, concatenation, and replication |
| 2.2 | Apply bitwise, arithmetic, and reduction operators correctly |
| 2.3 | Build multiplexers using the conditional operator |
| 2.4 | Compose modules hierarchically using named port connections |
| 2.5 | Design a hex-to-7-segment decoder targeting the Go Board display |
| 2.6 | Use properly sized literals to avoid width mismatch warnings |

## Exercises

### Exercise 1: Vector Operations Warm-Up (20 min)
Reduction operators on a 4-bit vector → LED display. Fill in `starter/w1d2_ex1_vector_ops.v`.

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex1_vector_ops.v
// =============================================================================
// Exercise 1: Vector Operations Warm-Up
// Day 2 · Combinational Building Blocks
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Use the 4 switches as a 4-bit input, display results on LEDs.
//
// LED1 = OR  reduction (any switch pressed)
// LED2 = AND reduction (all switches pressed)
// LED3 = XOR reduction (odd parity — odd number pressed)
// LED4 = MSB of the 4-bit value (switch1)
// =============================================================================

module vector_ops (
    input  wire i_switch1,
    input  wire i_switch2,
    input  wire i_switch3,
    input  wire i_switch4,
    output wire o_led1,
    output wire o_led2,
    output wire o_led3,
    output wire o_led4
);

    // Collect switches into a vector and make active-high
    // i_switch1 is MSB (bit 3), i_switch4 is LSB (bit 0)
    wire [3:0] w_sw;
    assign w_sw = ~{i_switch1, i_switch2, i_switch3, i_switch4};

    // TODO: Implement the four LED assignments
    // Remember: LEDs are active-low on the Go Board
    // Use the ~() inversion-at-output pattern

    // LED1: OR reduction — any switch pressed?
    // Hint: |w_sw gives OR reduction
    assign o_led1 = 1'b1;  // TODO: replace

    // LED2: AND reduction — all switches pressed?
    // Hint: &w_sw gives AND reduction
    assign o_led2 = 1'b1;  // TODO: replace

    // LED3: XOR reduction — odd number of switches pressed?
    // Hint: ^w_sw gives XOR reduction
    assign o_led3 = 1'b1;  // TODO: replace

    // LED4: MSB (switch1 state)
    assign o_led4 = 1'b1;  // TODO: replace

endmodule

### Exercise 2: 2:1 → 4:1 Multiplexer (25 min)
Build a 4:1 mux from three 2:1 mux instances. Fill in `starter/w1d2_ex2_mux4to1.v` and `starter/w1d2_ex2_top_mux.v`. Use `make ex2_show` to visualize the netlist in Yosys.

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex2_mux2to1.v
// =============================================================================
// Exercise 2, Part A: 2:1 Multiplexer
// Day 2 · Combinational Building Blocks
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================

module mux2to1 (
    input  wire i_a,
    input  wire i_b,
    input  wire i_sel,
    output wire o_y
);

    // sel=0 -> a, sel=1 -> b
    assign o_y = i_sel ? i_b : i_a;

endmodule

In [ ]:
%%writefile ex2_mux4to1.v
// =============================================================================
// Exercise 2, Part B: 4:1 Multiplexer from 2:1 Muxes
// Day 2 · Combinational Building Blocks
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Build a 4:1 mux by instantiating three 2:1 muxes hierarchically.
//
//   d0 ─┐
//        mux_lo ─┐
//   d1 ─┘        │
//                 mux_final ── o_y
//   d2 ─┐        │
//        mux_hi ─┘
//   d3 ─┘
//
//   sel[0] controls mux_lo and mux_hi
//   sel[1] controls mux_final
// =============================================================================

module mux4to1 (
    input  wire       i_d0,
    input  wire       i_d1,
    input  wire       i_d2,
    input  wire       i_d3,
    input  wire [1:0] i_sel,
    output wire       o_y
);

    wire w_mux_lo, w_mux_hi;

    // TODO: Instantiate three mux2to1 modules using NAMED port connections
    //
    // mux_lo:    selects between d0 and d1 using sel[0]
    // mux_hi:    selects between d2 and d3 using sel[0]
    // mux_final: selects between mux_lo and mux_hi using sel[1]

endmodule

In [ ]:
%%writefile ex2_top_mux.v
// =============================================================================
// Exercise 2, Part C: Top Module for Mux on Go Board
// Day 2 · Combinational Building Blocks
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================

module top_mux (
    input  wire i_switch1,   // sel[1]
    input  wire i_switch2,   // sel[0]
    input  wire i_switch3,   // data input (active-low)
    input  wire i_switch4,   // data input (active-low)
    output wire o_led1       // mux output
);

    // Invert switches at boundary
    wire [1:0] w_sel = ~{i_switch1, i_switch2};

    // Hardcode two data inputs, use sw3/sw4 for the other two
    wire w_d0 = 1'b0;          // fixed low
    wire w_d1 = ~i_switch3;    // from button 3
    wire w_d2 = ~i_switch4;    // from button 4
    wire w_d3 = 1'b1;          // fixed high

    wire w_result;

    // TODO: Instantiate mux4to1 here
    // mux4to1 mux (
    //     .i_d0(w_d0), .i_d1(w_d1), .i_d2(w_d2), .i_d3(w_d3),
    //     .i_sel(w_sel),
    //     .o_y(w_result)
    // );

    assign o_led1 = ~w_result;  // active-low output

endmodule

### Exercise 3: 4-Bit Ripple-Carry Adder (25 min)
Chain four `full_adder` modules. Fill in `starter/w1d2_ex3_ripple_adder_4bit.v` and `starter/w1d2_ex3_top_adder.v`.

#### 📝 Exercise 3 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex3_full_adder.v
// =============================================================================
// Exercise 3, Part A: Full Adder
// Day 2 · Combinational Building Blocks
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================

module full_adder (
    input  wire i_a,
    input  wire i_b,
    input  wire i_cin,
    output wire o_sum,
    output wire o_cout
);

    assign o_sum  = i_a ^ i_b ^ i_cin;
    assign o_cout = (i_a & i_b) | (i_a & i_cin) | (i_b & i_cin);

endmodule

In [ ]:
%%writefile ex3_ripple_adder_4bit.v
// =============================================================================
// Exercise 3, Part B: 4-Bit Ripple-Carry Adder
// Day 2 · Combinational Building Blocks
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Instantiate four full_adder modules to create a 4-bit adder.
// The carry output of each stage feeds into the carry input of the next.
// =============================================================================

module ripple_adder_4bit (
    input  wire [3:0] i_a,
    input  wire [3:0] i_b,
    input  wire       i_cin,
    output wire [3:0] o_sum,
    output wire       o_cout
);

    wire [3:1] w_carry;  // internal carry chain

    // TODO: Instantiate four full_adder modules
    // Use named port connections!
    //
    // full_adder fa0 (.i_a(i_a[0]), .i_b(i_b[0]), .i_cin(i_cin),       .o_sum(o_sum[0]), .o_cout(w_carry[1]));
    // full_adder fa1 ( ... );
    // full_adder fa2 ( ... );
    // full_adder fa3 (.i_a(i_a[3]), .i_b(i_b[3]), .i_cin(w_carry[3]),  .o_sum(o_sum[3]), .o_cout(o_cout));

endmodule

In [ ]:
%%writefile ex3_top_adder.v
// =============================================================================
// Exercise 3, Part C: Top Module — Adder on Go Board
// Day 2 · Combinational Building Blocks
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// sw1,sw2 = 2-bit input a   sw3,sw4 = 2-bit input b
// LEDs show the 3-bit sum (2 bits + carry)
// =============================================================================

module top_adder (
    input  wire i_switch1,
    input  wire i_switch2,
    input  wire i_switch3,
    input  wire i_switch4,
    output wire o_led1,     // sum[2] (carry into bit 2)
    output wire o_led2,     // sum[1]
    output wire o_led3,     // sum[0]
    output wire o_led4      // carry out
);

    // TODO: Invert switches, pad to 4 bits, instantiate adder, drive LEDs
    // Hint: wire [3:0] w_a = {2'b00, ~i_switch1, ~i_switch2};

endmodule

### Exercise 4: Hex-to-7-Segment Decoder (30 min)
Complete the nested conditional decoder in `starter/w1d2_ex4_hex_to_7seg.v`. Wire up the top module. Cycle through all 16 button combinations and verify each hex digit displays correctly.

#### 📝 Exercise 4 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex4_hex_to_7seg.v
// =============================================================================
// Exercise 4: Hex-to-7-Segment Decoder
// Day 2 · Combinational Building Blocks
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Input:  4-bit hex value (0-F)
// Output: 7 segment signals (active low for Go Board)
//         o_seg = {a, b, c, d, e, f, g}
//
// Segment layout:
//     ─a─
//    |   |
//    f   b
//    |   |
//     ─g─
//    |   |
//    e   c
//    |   |
//     ─d─
//
// Active low: 0 = segment on, 1 = segment off
// =============================================================================

module hex_to_7seg (
    input  wire [3:0] i_hex,
    output wire [6:0] o_seg   // {a, b, c, d, e, f, g} — active low
);

    // TODO: Implement using nested conditional operator
    // Each 7-bit pattern encodes which segments are on (0) or off (1)
    //
    //                              abcdefg
    // 0: segments a,b,c,d,e,f on = 0000001
    // 1: segments b,c on         = 1001111
    // 2: segments a,b,d,e,g on   = 0010010
    // ... complete all 16 values (0-F)

    assign o_seg = (i_hex == 4'h0) ? 7'b0000001 :  // 0
                   (i_hex == 4'h1) ? 7'b1001111 :  // 1
                   // TODO: Complete entries for 2-E
                                     7'b0111000 ;  // F (default/last)

endmodule

In [ ]:
%%writefile ex4_top_7seg.v
// =============================================================================
// Exercise 4: Top Module — 7-Seg Display on Go Board
// Day 2 · Combinational Building Blocks
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================

module top_7seg (
    input  wire i_switch1,
    input  wire i_switch2,
    input  wire i_switch3,
    input  wire i_switch4,
    output wire o_segment1_a,
    output wire o_segment1_b,
    output wire o_segment1_c,
    output wire o_segment1_d,
    output wire o_segment1_e,
    output wire o_segment1_f,
    output wire o_segment1_g
);

    // Collect switches into 4-bit hex value (active-high internally)
    wire [3:0] w_hex = ~{i_switch1, i_switch2, i_switch3, i_switch4};

    // Decoder output
    wire [6:0] w_seg;

    // TODO: Instantiate hex_to_7seg decoder
    // hex_to_7seg decoder (
    //     .i_hex(w_hex),
    //     .o_seg(w_seg)
    // );

    // Map decoder output to individual segment pins
    assign o_segment1_a = w_seg[6];
    assign o_segment1_b = w_seg[5];
    assign o_segment1_c = w_seg[4];
    assign o_segment1_d = w_seg[3];
    assign o_segment1_e = w_seg[2];
    assign o_segment1_f = w_seg[1];
    assign o_segment1_g = w_seg[0];

endmodule

### Exercise 5 — Stretch: Adder + Display Integration (20 min)
Combine the adder and decoder into a single design that displays the sum on 7-seg.

#### 📝 Exercise 5 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex5_top_adder_display.v
// =============================================================================
// Exercise 5 (Stretch): Adder + 7-Seg Display Integration
// Day 2 · Combinational Building Blocks
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Combine the adder with the 7-seg decoder.
// sw1,sw2 = 2-bit a    sw3,sw4 = 2-bit b
// 7-seg shows the sum as a hex digit
// =============================================================================

module top_adder_display (
    input  wire i_switch1,
    input  wire i_switch2,
    input  wire i_switch3,
    input  wire i_switch4,
    output wire o_led1,          // carry out
    output wire o_segment1_a,
    output wire o_segment1_b,
    output wire o_segment1_c,
    output wire o_segment1_d,
    output wire o_segment1_e,
    output wire o_segment1_f,
    output wire o_segment1_g
);

    // TODO: 
    // 1. Invert switches, create 4-bit padded inputs
    // 2. Instantiate ripple_adder_4bit
    // 3. Instantiate hex_to_7seg with the sum
    // 4. Map segment outputs to pins
    // 5. Drive o_led1 from carry (active-low)

endmodule

## Deliverable Checklist

- [ ] Exercise 1: LEDs respond correctly to reduction operations
- [ ] Exercise 2: 4:1 mux works on board
- [ ] Exercise 3: Adder shows correct sums on LEDs
- [ ] Exercise 4: All 16 hex digits display on 7-segment
- [ ] At minimum: Exercise 4 (hex display) programmed and working

## Quick Reference

In [ ]:
!make ex1          make ex2          make ex3

In [ ]:
!make ex4          make ex5          make ex2_show

In [ ]:
!make clean